# Coupang Competitor Brand Review Analysis
Analyzes Coupang product reviews for competitor diaper brands.
Extracts satisfaction/dissatisfaction keywords, generates property scores and summaries,
and flags risk-related mentions using GPT-4.

**Same structure as internal brand analysis but targets competitor brands.**

**Run:** Trigger by updating `start_date` and `end_date` at the top.

In [ ]:
# pip install googletrans==4.0.0rc1 snowflake-connector-python snowflake-sqlalchemy==1.5.1 openai

In [ ]:
import os
import ast
import time
import pandas as pd
import openai
import snowflake.connector
from datetime import datetime, timedelta
from googletrans import Translator
from snowflake.sqlalchemy import URL
from sqlalchemy import create_engine
from snowflake.connector.pandas_tools import pd_writer

# Date range for this batch run (update before each run)
start_date = (datetime.now() - timedelta(days=15)).strftime('%Y-%m-%d')
end_date   = (datetime.now() - timedelta(days=9)).strftime('%Y-%m-%d')
print(f'Processing: {start_date} to {end_date}')

In [ ]:
# ── DATABASE CONNECTION ──────────────────────────────────────────────────
# Set environment variables before running:
#   SNOWFLAKE_USER, SNOWFLAKE_PASSWORD, SNOWFLAKE_ACCOUNT
#   SNOWFLAKE_DATABASE, SNOWFLAKE_SCHEMA, SNOWFLAKE_WAREHOUSE
#   OPENAI_API_KEY

con = snowflake.connector.connect(
    user=os.environ['SNOWFLAKE_USER'],
    password=os.environ['SNOWFLAKE_PASSWORD'],
    account=os.environ['SNOWFLAKE_ACCOUNT'],
    database=os.environ['SNOWFLAKE_DATABASE'],
    schema=os.environ['SNOWFLAKE_SCHEMA'],
    warehouse=os.environ['SNOWFLAKE_WAREHOUSE']
)
cur = con.cursor()

## Part 1: Keyword Extraction

In [ ]:
# ── LOAD COMPETITOR REVIEW DATA ──────────────────────────────────────────

review_query = f"""
SELECT DATE, max(BRAND), max(SUBBRAND), REVIEW
FROM COMPETITOR_REVIEW_TABLE
WHERE DATE >= '{start_date}' AND DATE <= '{end_date}'
GROUP BY DATE, REVIEW, PROPERTY;
"""
review = pd.DataFrame(cur.execute(review_query))
review.columns = ['DATE', 'BRAND', 'SUBBRAND', 'REVIEW']

In [ ]:
# ── GPT-4 KEYWORD EXTRACTION ─────────────────────────────────────────────

openai.api_key = os.environ['OPENAI_API_KEY']
GPT_MODEL = 'gpt-4-turbo-2024-04-09'

COMPETITOR_BRANDS = ['Pampers', 'Penelope', 'Mamipoko', 'Bosomi', 'Kindoh', 'NatureLoveMere']

def extract_keywords(reviews, brand):
    """Extract top 3 satisfaction and dissatisfaction keywords from a batch of competitor reviews."""
    prompt = (
        f'Reviews: {reviews} '
        'Prompt: The above are diaper reviews. Excluding delivery and sizing, '
        'identify 3 keywords each for satisfaction and dissatisfaction. '
        'Output format: {"satisfied": keywords, "unsatisfied": keywords}. No other output.'
    )
    messages = [
        {'role': 'system', 'content': f'You are a marketer for the {brand} diaper brand.'},
        {'role': 'user', 'content': prompt}
    ]
    response = openai.ChatCompletion.create(model=GPT_MODEL, messages=messages)
    return response.choices[0].message.content

In [ ]:
# ── BATCH EXTRACTION ─────────────────────────────────────────────────────

col_brand, col_word, col_sentiment, col_date = [], [], [], []
dates = list(review.sort_values('DATE')['DATE'].unique())

for brand in COMPETITOR_BRANDS:
    for date_ in dates:
        rev = review[(review['BRAND'] == brand) & (review['DATE'] == date_)]
        for i in range(int((len(rev) + 2) / 3)):
            try:
                str_d = extract_keywords(rev['REVIEW'][i*3:(i+1)*3].values, brand)
                d = ast.literal_eval(str_d)
                for sentiment in ['satisfied', 'unsatisfied']:
                    for keyword in d.get(sentiment, []):
                        col_brand.append(brand)
                        col_word.append(keyword)
                        col_sentiment.append(sentiment)
                        col_date.append(date_)
                print(f'{date_} | {brand}')
            except Exception as e:
                print(f'Failed: {e}')
                time.sleep(60)

output = pd.DataFrame({
    'DATE': pd.to_datetime(col_date, format='%Y-%m-%d').astype(str),
    'BRAND': col_brand,
    'KEYWORD': col_word,
    'SENTIMENT': col_sentiment,
    'START_DT': start_date,
    'END_DT': end_date
})

In [ ]:
# ── TRANSLATE AND UPLOAD KEYWORDS ────────────────────────────────────────

translator = Translator()
translation_map = {}

for keyword in output['KEYWORD'].unique():
    try:
        if keyword:
            translation_map[keyword] = translator.translate(keyword, src='ko', dest='en').text
        time.sleep(0.3)
    except Exception:
        continue

output['KEYWORD_EN'] = output['KEYWORD'].map(translation_map)
output['SENTIMENT_EN'] = output['SENTIMENT']

engine = create_engine(URL(
    account=os.environ['SNOWFLAKE_ACCOUNT'],
    user=os.environ['SNOWFLAKE_USER'],
    password=os.environ['SNOWFLAKE_PASSWORD'],
    database=os.environ['SNOWFLAKE_DATABASE'],
    schema=os.environ['SNOWFLAKE_SCHEMA'],
    warehouse=os.environ['SNOWFLAKE_WAREHOUSE'],
))

with engine.connect() as con:
    output.to_sql(name='COMPETITOR_KEYWORDS_TABLE', con=con, if_exists='append',
                  method=pd_writer, index=False)

print(f'Uploaded {len(output)} competitor keyword records')

## Part 2: Property Rating + Summary

In [ ]:
# ── LOAD KEYWORD DATA ────────────────────────────────────────────────────

df = pd.DataFrame(cur.execute(
    f"SELECT * FROM COMPETITOR_KEYWORDS_TABLE "
    f"WHERE DATE >= '{start_date}' AND DATE <= '{end_date}'"
))
df = df.rename(columns={1: 'date', 2: 'brand', 3: 'keyword', 4: 'sentiment'})
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d').astype(str)

In [ ]:
# ── SCORING AND SUMMARY ──────────────────────────────────────────────────

PROPERTIES_KO = ['흡수력', '부드러움', '편함', '가격', '피부 저자극']
PROPERTY_MAP = {
    '흡수력': 'Absorbency', '부드러움': 'Softness', '편함': 'Comfort',
    '가격': 'Price', '피부 저자극': 'Hypoallergenic'
}

def make_score(brand, pros, cons):
    prompt = (
        f'Pros: {pros}\nCons: {cons}\n'
        f'Based on these keywords, score each of {PROPERTIES_KO} out of 10. '
        'Return only a list of 5 scores. No other output.'
    )
    messages = [
        {'role': 'system', 'content': f'You are a marketer for {brand}.'},
        {'role': 'user', 'content': prompt}
    ]
    return openai.ChatCompletion.create(model=GPT_MODEL, messages=messages).choices[0].message.content

def make_summary(brand, pros, cons):
    prompt = (
        f'Pros: {pros}\nCons: {cons}\n'
        'Summarize consumer opinion in 2 Korean sentences based on keyword frequency.'
    )
    messages = [
        {'role': 'system', 'content': f'You are a marketer for {brand}.'},
        {'role': 'user', 'content': prompt}
    ]
    return openai.ChatCompletion.create(model=GPT_MODEL, messages=messages).choices[0].message.content

rating_output = pd.DataFrame()
for brand in COMPETITOR_BRANDS:
    target = df[(df['date'] >= start_date) & (df['date'] <= end_date) & (df['brand'] == brand)]
    pros = target[target['sentiment'] == 'satisfied']['keyword'].tolist()
    cons = target[target['sentiment'] == 'unsatisfied']['keyword'].tolist()
    summary_text = make_summary(brand, pros, cons).replace("'", '').replace('"', '')
    scores_raw = make_score(brand, pros, cons)
    scores = list(map(float, scores_raw.replace('[','').replace(']','').split(',')))
    for i, prop in enumerate(PROPERTIES_KO):
        score_val = scores[i] if scores[i] != 0 else sum(scores) / len(scores)
        rating_output = rating_output.append({
            'START_DT': start_date, 'END_DT': end_date,
            'BRAND': brand, 'PROPERTY': prop,
            'PROPERTY_EN': PROPERTY_MAP[prop],
            'RATING': score_val, 'SUMMARY': summary_text
        }, ignore_index=True)

In [ ]:
# ── TRANSLATE SUMMARIES AND UPLOAD ───────────────────────────────────────

summary_map = {}
for s in rating_output['SUMMARY'].unique():
    try:
        if s:
            summary_map[s] = translator.translate(s, src='ko', dest='en').text
        time.sleep(0.3)
    except Exception:
        continue

rating_output['SUMMARY_EN'] = rating_output['SUMMARY'].map(summary_map)
rating_output = rating_output.sort_values('START_DT')

with engine.connect() as con:
    rating_output.to_sql(name='COMPETITOR_RATING_TABLE', con=con, if_exists='append',
                         method=pd_writer, index=False)
print(f'Uploaded {len(rating_output)} competitor rating records')

## Part 3: Risk Keyword Detection

In [ ]:
# ── RISK DETECTION FOR COMPETITOR REVIEWS ────────────────────────────────

RISK_KEYWORDS_KO = [
    '유해물질','발암','휘발성유기화합물','VOC','벤젠','톨루엔',
    '스티렌','자일렌','독성물질','트리클로로에틸렌','유해성','에틸벤젠',
    '총휘발성유기화합물','TVOC','이물질','녹물','본드','곰팡이','벌레','쇳가루','금속'
]

risk_review = pd.DataFrame(cur.execute(
    f"SELECT DATE, max(BRAND), max(SUBBRAND), REVIEW "
    f"FROM COMPETITOR_REVIEW_TABLE "
    f"WHERE DATE >= '{start_date}' AND DATE <= '{end_date}' "
    f"GROUP BY DATE, REVIEW;"
))
risk_review.columns = ['DATE', 'BRAND', 'SUBBRAND', 'REVIEW']

def detect_risk_keywords(review_text):
    return ', '.join(kw for kw in RISK_KEYWORDS_KO if kw in review_text)

risk_review['RISK_KEYWORD'] = risk_review['REVIEW'].apply(detect_risk_keywords)

review_translation = {}
for text in risk_review['REVIEW'].unique():
    try:
        if text:
            review_translation[text] = translator.translate(text, src='ko', dest='en').text
        time.sleep(0.3)
    except Exception:
        continue

risk_review['RISK_KEYWORD_EN'] = risk_review['RISK_KEYWORD'].apply(
    lambda x: translator.translate(x, src='ko', dest='en').text if x else ''
)
risk_review['REVIEW_EN'] = risk_review['REVIEW'].apply(
    lambda x: review_translation.get(x, 'Translation unavailable')
)
risk_review['DATE'] = pd.to_datetime(risk_review['DATE'], format='%Y%m%d').astype(str)
risk_review['START_DT'] = start_date
risk_review['END_DT'] = end_date

In [ ]:
# ── UPLOAD RISK DATA ─────────────────────────────────────────────────────

with engine.connect() as con:
    risk_review.to_sql(name='COMPETITOR_RISK_TABLE', con=con, if_exists='append',
                       method=pd_writer, index=False)

print(f'Uploaded {len(risk_review)} competitor risk records')